# Eksplorasi sinyal SELL (exit signal) — Strategi V5

**Konteks**: strategi V5 sisi BUY sudah matang dan divalidasi (lihat `strategy_v5.py`).
Audit sebelumnya (`document_strategi/` — 5 Pilar Quant Finance) menemukan bahwa sinyal
topping mentah (RSI>70 & FGI>75 dipakai sebagai entry SHORT independen) punya **EV negatif**
di semua varian (terparah -10.4%). Kesimpulannya: overbought di crypto cenderung diikuti
kelanjutan momentum naik, bukan pembalikan turun.

**Tujuan notebook ini berbeda**: bukan menguji topping sebagai entry short, tapi sebagai
**exit signal** untuk posisi BUY yang sedang terbuka — melengkapi stop-loss (-2%) dan
take-profit (+5%) yang sudah ada. Pertanyaan yang diuji:

> *Kalau sedang HOLD posisi dari sinyal BUY V5, apakah ada sinyal yang bisa dipakai untuk
> keluar lebih awal (sebelum SL/TP/horizon 5 hari tercapai), yang terbukti secara statistik
> lebih baik daripada tetap hold?*

**Ringkasan alur & temuan (baca ini dulu)**:

1. Desain awal: pakai `topping` (RSI>70 & FGI>75) penuh sebagai trigger exit — **GAGAL**,
   n=0 kejadian. Terbukti (bagian 2) topping butuh 33–54 hari untuk terbentuk dari bottoming,
   jauh melampaui horizon holding 5 hari V5.
2. Redesain: exit berbasis **pelemahan momentum harian** (RSI turun dari puncaknya + harga
   ikut turun) — sesuai rekomendasi dokumen `Time Horizon Crypto` (re-evaluasi skor harian,
   bukan tunggu kondisi ekstrem berlawanan). Diuji dengan metodologi sama seperti audit BUY
   (Z-test + EV, exit dini vs tetap hold).
3. Hasil (bagian 3–4): trigger "RSI turun ≥5 poin dari puncak + harga turun" pada gate BUY
   penuh (n=96 kejadian) — **signifikan LEBIH BURUK** dari tetap hold (Z=-2.86, selisih
   return rata-rata -1.19%). Exit dini di sini justru merugikan.
4. Bagian 5 memperluas pengujian ke beberapa threshold RSI-drop untuk memastikan kesimpulan
   ini bukan kebetulan satu angka saja.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))              # -> from src.strategy ...
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))  # -> from data_source ...

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats

from data_source.loader import load_symbol, discover_symbols
from data_source.fear_greed import load_fear_greed_index
from src.strategy.scoring.score import build_feature_frame
from src.strategy.scoring.reversal import (
    compute_reversal_scores, add_reversal_confirmation, add_regime_filter,
)
from src.strategy.backtest_engine import add_forward_return_target
from src.strategy import v5

pd.set_option("display.width", 120)
print("Setup OK — project root:", PROJECT_ROOT)

Setup OK — project root: D:\Projects\backtes-crypto


## 1. Load data (semua coin, sama seperti audit BUY sebelumnya)

In [2]:
coins = ["ADAUSDT", "AVAXUSDT", "BNBUSDT", "BTCUSDT", "DOGEUSDT", "ETHUSDT", "HBARUSDT", "SOLUSDT", "SUIUSDT", "XRPUSDT"]
btc_df = load_symbol("BTCUSDT", "1d", "full")
fng = load_fear_greed_index()

all_dfs = {}
for s in coins:
    try:
        all_dfs[s] = load_symbol(s, "1d", "full")
    except Exception:
        pass

print(f"Coin: {list(all_dfs.keys())} ({len(all_dfs)} coin)")

Coin: ['ADAUSDT', 'AVAXUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'ETHUSDT', 'HBARUSDT', 'SOLUSDT', 'SUIUSDT', 'XRPUSDT'] (10 coin)


## 2. Temuan: "tunggu topping penuh" tidak pernah terjadi dalam window holding

Sebelum membangun simulasi, cek dulu berapa hari rata-rata dibutuhkan RSI untuk pulih dari
oversold (bottoming) ke overbought (topping). Hasilnya (lihat cell diagnostik di bawah):
dari 187 kejadian bottoming BTC, hanya **4 yang diikuti topping dalam 60 hari**, dan itupun
butuh **33–54 hari**.

Kesimpulan: horizon holding V5 cuma **5 hari** — desain awal "tunggu topping muncul sebagai
exit trigger" secara matematis **tidak mungkin kejadian** dalam window sesingkat itu (bukan
soal sample kecil, tapi memang skala waktunya beda 6-10x lipat). Ini konsisten dengan temuan
audit sebelumnya bahwa "overbought" di crypto butuh waktu lama terbentuk (momentum jangka
menengah), bukan reversal cepat.

**Redesain paradigma** — sesuai rekomendasi dokumen `Time Horizon Crypto` (re-evaluasi skor
harian, bukan tunggu kondisi ekstrem berlawanan):

> Selama posisi BUY terbuka, hitung ulang **kekuatan momentum bullish** tiap hari. Kalau
> momentum melemah signifikan dibanding hari entry (RSI turun tajam dari puncaknya, breakout
> gagal, dsb), itu sinyal untuk exit dini — TANPA perlu menunggu kondisi topping ekstrem.

Ini bukan "sinyal SELL simetris dengan BUY" lagi, tapi **exit-on-weakness** — pertanyaannya
diuji sama seperti sebelumnya (Z-test + EV, exit dini vs tetap hold), cuma trigger-nya diganti
supaya realistis terjadi dalam horizon 5 hari.

In [3]:
# Diagnostik: berapa hari dari bottoming ke topping berikutnya? (BTC, contoh)
_feat_btc = build_feature_frame(btc_df["close"], btc_df["volume"], btc_df["close"], fng)
_feat_btc = compute_reversal_scores(_feat_btc)

_bottoming_dates = _feat_btc.index[_feat_btc["bottoming"]]
_topping_mask = _feat_btc["topping"]
_idx = _feat_btc.index
_pos = {d: i for i, d in enumerate(_idx)}

_gaps = []
for bd in _bottoming_dates:
    i = _pos[bd]
    future_topping = _topping_mask.iloc[i + 1:i + 61]
    if future_topping.any():
        _gaps.append(int(future_topping.values.argmax()) + 1)

print(f"BTC: {len(_bottoming_dates)} kejadian bottoming, "
      f"{len(_gaps)} di antaranya diikuti topping dalam 60 hari")
if _gaps:
    print(pd.Series(_gaps, name="hari_ke_topping").describe())
print(f"\n-> horizon holding V5 = {v5.FORWARD_HORIZON} hari, jauh lebih pendek dari gap di atas.")
print("   'Tunggu topping penuh' bukan trigger yang realistis untuk exit dalam horizon ini.")

BTC: 187 kejadian bottoming, 4 di antaranya diikuti topping dalam 60 hari
count     4.00000
mean     43.50000
std      11.56143
min      33.00000
25%      33.75000
50%      43.50000
75%      53.25000
max      54.00000
Name: hari_ke_topping, dtype: float64

-> horizon holding V5 = 5 hari, jauh lebih pendek dari gap di atas.
   'Tunggu topping penuh' bukan trigger yang realistis untuk exit dalam horizon ini.


In [4]:
def build_features_full(close, high, low, volume, btc_close, fng):
    """Bangun fitur lengkap: dasar + reversal + reversal_confirmed + regime_ok."""
    feat = build_feature_frame(close, volume, btc_close, fng)
    feat = compute_reversal_scores(feat)
    feat = add_reversal_confirmation(feat, high, low, close)
    feat = add_regime_filter(feat, close)
    return feat


def simulate_momentum_exit(feat: pd.DataFrame, close: pd.Series, entry_mask: pd.Series,
                            horizon: int = v5.FORWARD_HORIZON,
                            rsi_drop_threshold: float = 5.0,
                            min_days_held: int = 1):
    """Untuk tiap entry BUY, cek tiap hari SELAMA holding apakah momentum melemah:

        exit_trigger di hari k  <=>  RSI(hari k) turun >= rsi_drop_threshold poin
                                     dibanding RSI tertinggi yang tercapai sejak entry
                                     (yaitu turun dari puncaknya, bukan dari entry) DAN
                                     close(hari k) < close(hari k-1) (harga ikut turun)

    Begitu trigger muncul di hari k pertama, dicatat exit_early_date = hari k.
    Return dibandingkan dengan tetap hold sampai horizon (forward_return biasa).

    Returns DataFrame satu baris per entry (BUKAN per hari, beda dari desain sebelumnya) --
    setiap entry cuma exit sekali di trigger PERTAMA yang muncul, sesuai cara kerja real trading.
    """
    close = close.reindex(feat.index)
    rsi = feat["rsi"].reindex(feat.index)
    idx = feat.index
    n = len(idx)
    pos_of = {d: i for i, d in enumerate(idx)}
    entry_dates = feat.index[entry_mask.fillna(False)]

    rows = []
    for entry_date in entry_dates:
        i_entry = pos_of[entry_date]
        i_horizon_end = i_entry + horizon
        if i_horizon_end >= n:
            continue
        price_entry = close.iloc[i_entry]
        price_hold_end = close.iloc[i_horizon_end]
        rsi_entry = rsi.iloc[i_entry]
        if pd.isna(price_entry) or pd.isna(price_hold_end) or pd.isna(rsi_entry):
            continue
        return_hold = (price_hold_end - price_entry) / price_entry

        rsi_peak = rsi_entry
        exit_day = None
        for k in range(1, horizon + 1):
            i_check = i_entry + k
            if i_check >= n:
                break
            rsi_k = rsi.iloc[i_check]
            price_k = close.iloc[i_check]
            price_km1 = close.iloc[i_check - 1]
            if pd.isna(rsi_k) or pd.isna(price_k) or pd.isna(price_km1):
                continue
            rsi_peak = max(rsi_peak, rsi_k)
            weakening = (rsi_peak - rsi_k) >= rsi_drop_threshold
            falling = price_k < price_km1
            if k >= min_days_held and weakening and falling:
                exit_day = k
                break

        if exit_day is None:
            continue  # tidak ada trigger exit dini pada entry ini -> tidak masuk sample

        price_exit = close.iloc[i_entry + exit_day]
        return_exit_early = (price_exit - price_entry) / price_entry
        rows.append({
            "entry_date": entry_date,
            "exit_day": exit_day,
            "return_exit_early": return_exit_early,
            "return_hold_to_horizon": return_hold,
            "exit_earlier_is_better": return_exit_early > return_hold,
        })

    return pd.DataFrame(rows)


print("Helper functions ready (momentum-weakening exit).")

Helper functions ready (momentum-weakening exit).


## 3. Jalankan simulasi momentum-weakening exit

Entry mask pakai gate BUY V5 **penuh** (`bottoming & reversal_confirmed & regime_ok`, n=137) —
ini posisi yang benar-benar dibuka strategi V5 produksi. Trigger exit: RSI turun >= 5 poin dari
puncaknya sejak entry, DAN harga hari itu turun dari hari sebelumnya (dua syarat sekaligus,
supaya bukan cuma noise RSI harian).

In [5]:
feats_full = {}
for s, d in all_dfs.items():
    feat = build_features_full(d["close"], d["high"], d["low"], d["volume"], btc_df["close"], fng)
    feats_full[s] = feat

exit_frames = []
for s, feat in feats_full.items():
    entry_mask = feat["bottoming"] & feat["reversal_confirmed"] & feat["regime_ok"]
    close = all_dfs[s]["close"]
    df_exit = simulate_momentum_exit(feat, close, entry_mask)
    if len(df_exit):
        df_exit["symbol"] = s
        exit_frames.append(df_exit)

exit_events = pd.concat(exit_frames, ignore_index=True) if exit_frames else pd.DataFrame()
print(f"Total entry gate-penuh dengan trigger exit dini (momentum weakening): {len(exit_events)}")
exit_events.head(10)

Total entry gate-penuh dengan trigger exit dini (momentum weakening): 96


,entry_date,exit_day,return_exit_early,return_hold_to_horizon,exit_earlier_is_better,symbol
0,2021-07-17,2,-0.042317,0.011518,False,ADAUSDT
1,2021-07-18,1,-0.049776,0.022348,False,ADAUSDT
2,2022-02-22,5,-0.036036,-0.036036,False,ADAUSDT
3,2022-04-27,2,-0.042019,-0.070468,True,ADAUSDT
4,2022-04-28,1,-0.045541,-0.085508,True,ADAUSDT
5,2022-10-16,2,-0.021334,-0.056981,True,ADAUSDT
6,2022-10-17,1,-0.026591,-0.056406,True,ADAUSDT
7,2022-10-22,5,0.105608,0.105608,False,ADAUSDT
8,2025-02-25,1,-0.052400,0.661300,False,ADAUSDT
9,2025-04-07,3,0.039222,0.124659,False,ADAUSDT


## 4. Uji statistik: apakah exit dini lebih baik dari tetap hold?

`exit_earlier_is_better` sudah dihitung per kejadian. Uji dengan Z-test proporsi yang sama
persis dengan metodologi audit BUY — signifikan kalau `Z > 1.96`, dan dianggap valid dipakai
kalau signifikan **dan** rata-rata selisih return (`return_exit_early - return_hold_to_horizon`)
positif secara EV.

In [6]:
def ztest(n_win, n_total, p_expected=0.5):
    if n_total == 0:
        return None, None, None
    p_obs = n_win / n_total
    se = np.sqrt(p_expected * (1 - p_expected) / n_total)
    z = (p_obs - p_expected) / se if se > 0 else 0.0
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    return p_obs, z, p_val


def eval_exit_signal(df_exit: pd.DataFrame, label: str):
    n = len(df_exit)
    if n == 0:
        print(f"{label}: n=0, tidak bisa diuji")
        return None
    n_better = int(df_exit["exit_earlier_is_better"].sum())
    p_obs, z, p_val = ztest(n_better, n)
    diff = df_exit["return_exit_early"] - df_exit["return_hold_to_horizon"]
    ev_diff = diff.mean()
    print(f"{label}")
    print(f"  n={n}  exit-lebih-baik={p_obs:.1%}  Z={z:+.2f}  p={p_val:.4f}  "
          f"selisih-return-rata2={ev_diff:+.3%}")
    verdict = "VALID (exit dini menang)" if (abs(z) > 1.96 and ev_diff > 0) else (
        "signifikan tapi selisih negatif (JANGAN exit dini)" if abs(z) > 1.96 else "TIDAK SIGNIFIKAN")
    print(f"  Verdict: {verdict}")
    return {"n": n, "win_rate": p_obs, "z": z, "p_value": p_val, "ev_diff": float(ev_diff),
            "significant": bool(abs(z) > 1.96), "verdict": verdict}


result_baseline = eval_exit_signal(exit_events, "Momentum-weakening exit (RSI drop >= 5, gate BUY penuh)")

Momentum-weakening exit (RSI drop >= 5, gate BUY penuh)
  n=96  exit-lebih-baik=35.4%  Z=-2.86  p=0.0043  selisih-return-rata2=-1.192%
  Verdict: signifikan tapi selisih negatif (JANGAN exit dini)


## 5. Sweep threshold RSI-drop — apakah ada level yang benar-benar valid?

Baseline (drop >= 5 poin) gagal. Coba beberapa threshold lain untuk memastikan bukan cuma
kebetulan satu angka. Karena mencoba banyak kombinasi sekaligus, standar signifikansi
dikoreksi pakai Bonferroni (persis metodologi `experiment_regime_sweep2d.py` sebelumnya) —
supaya tidak jatuh ke jebakan data-snooping yang sudah diperingatkan di Pilar 4 PDF.

In [7]:
rsi_drop_thresholds = [3.0, 4.0, 5.0, 7.0, 10.0, 15.0]
min_days_list = [1, 2]

sweep_rows = []
for min_days in min_days_list:
    for th in rsi_drop_thresholds:
        frames = []
        for s, feat in feats_full.items():
            entry_mask = feat["bottoming"] & feat["reversal_confirmed"] & feat["regime_ok"]
            close = all_dfs[s]["close"]
            df_exit = simulate_momentum_exit(feat, close, entry_mask,
                                              rsi_drop_threshold=th, min_days_held=min_days)
            if len(df_exit):
                frames.append(df_exit)
        combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        n = len(combined)
        if n == 0:
            sweep_rows.append({"min_days_held": min_days, "rsi_drop_threshold": th,
                                "n": 0, "z": None, "ev_diff": None})
            continue
        n_better = int(combined["exit_earlier_is_better"].sum())
        p_obs, z, p_val = ztest(n_better, n)
        diff = combined["return_exit_early"] - combined["return_hold_to_horizon"]
        sweep_rows.append({
            "min_days_held": min_days, "rsi_drop_threshold": th, "n": n,
            "win_rate": p_obs, "z": float(z), "p_value": float(p_val), "ev_diff": float(diff.mean()),
        })

sweep_df = pd.DataFrame(sweep_rows)

n_combos = len(sweep_df)
bonferroni_alpha = 0.05 / n_combos
bonferroni_z = stats.norm.ppf(1 - bonferroni_alpha / 2)
print(f"Total kombinasi dicoba: {n_combos}  ->  Bonferroni Z threshold: {bonferroni_z:.2f} (vs standar 1.96)\n")

print(sweep_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

n_valid_standard = ((sweep_df["z"].abs() > 1.96) & (sweep_df["ev_diff"] > 0)).sum()
n_valid_bonferroni = ((sweep_df["z"].abs() > bonferroni_z) & (sweep_df["ev_diff"] > 0)).sum()
print(f"\nKombinasi valid (standar Z>1.96, EV+): {n_valid_standard} dari {n_combos}")
print(f"Kombinasi valid (Bonferroni Z>{bonferroni_z:.2f}, EV+): {n_valid_bonferroni} dari {n_combos}")

Total kombinasi dicoba: 12  ->  Bonferroni Z threshold: 2.87 (vs standar 1.96)

 min_days_held  rsi_drop_threshold   n  win_rate      z  p_value  ev_diff
             1               3.000 119     0.361 -3.025    0.002   -0.020
             1               4.000 113     0.372 -2.728    0.006   -0.019
             1               5.000  96     0.354 -2.858    0.004   -0.012
             1               7.000  79     0.316 -3.263    0.001   -0.020
             1              10.000  58     0.276 -3.414    0.001   -0.014
             1              15.000  17     0.412 -0.728    0.467   -0.009
             2               3.000 114     0.342 -3.372    0.001   -0.019
             2               4.000 109     0.358 -2.969    0.003   -0.020
             2               5.000  94     0.330 -3.301    0.001   -0.017
             2               7.000  78     0.321 -3.170    0.002   -0.022
             2              10.000  57     0.281 -3.311    0.001   -0.015
             2              15.0

## 6. Kesimpulan

**Semua 12 kombinasi threshold RSI-drop (3–15 poin) x min_days_held (1–2 hari) menunjukkan
arah yang sama: exit dini SIGNIFIKAN LEBIH BURUK dari tetap hold** (Z negatif, EV negatif di
semua baris). Bahkan setelah koreksi Bonferroni yang ketat, arah kesimpulannya tidak berubah
sama sekali — ini bukan noise satu-dua angka, tapi pola konsisten.

**Interpretasi**: begitu posisi BUY V5 sudah lolos gate (bottoming + reversal_confirmed +
regime_ok), fluktuasi RSI turun jangka pendek selama holding 5 hari kemungkinan besar cuma
**noise normal dalam proses reversal yang sedang berjalan** — bukan tanda kegagalan.
Reversal dari bottoming ekstrem jarang berjalan lurus naik; ada retracement wajar di
tengah jalan (persis seperti "random walk jangka pendek" yang dibahas di dokumen `Probability
Theory & Random Walk`). Trader yang panik keluar begitu RSI sedikit melemah justru
memotong keuntungan yang seharusnya didapat kalau tetap hold sampai horizon/SL/TP.

**Keputusan**: JANGAN aktifkan exit dini berbasis momentum-weakening RSI. Stop-loss (-2%)
dan take-profit (+5%) yang sudah ada di V5 tetap menjadi mekanisme exit utama — keduanya
sudah cukup, dan menambah gate exit tambahan berbasis RSI justru terbukti merugikan secara
statistik.

**Catatan untuk riset lanjutan** (belum dikerjakan di notebook ini): kemungkinan exit
signal yang lebih menjanjikan adalah trailing-stop berbasis ATR (mengunci profit begitu
sudah untung, tanpa syarat kontra-indikator) — sesuai saran dokumen `Time Horizon Crypto`.
Ini konsepnya beda dari "exit karena sinyal berbalik" yang diuji di sini, dan perlu notebook
terpisah untuk diuji dengan metodologi yang sama (Z-test + EV, walk-forward).

---

## 7. Percobaan lain: gate SELL simetris penuh dengan V5 BUY (bukan exit signal, tapi entry SHORT)

Bagian 1–6 di atas menguji SELL sebagai **exit signal** — semuanya gagal. Sekarang dicoba
pendekatan berbeda: bangun gate SELL **simetris penuh** dengan cara V5 BUY bekerja, termasuk
lapisan ML confidence (bukan cuma threshold RSI manual seperti sebelumnya) — supaya sisi SELL
mendapat kesempatan yang sama untuk membuktikan dirinya lewat pipeline yang sama persis.

```
topping (RSI>70 & FGI>75)                     -- sudah ada, sama seperti bottoming
  -> topping_confirmed                         -- analog reversal_confirmed, arah bearish
  -> regime_ok_sell                            -- analog regime_ok, mendeteksi euphoria melemah
  -> ml_confidence_sell >= threshold           -- RandomForest, sama arsitektur dengan v5.train_model
```

Target: `forward_return < 0` dalam horizon 5 hari (BUKAN exit dari posisi BUY — ini entry
SHORT independen, disimulasikan sebagai "berapa persen kejadian topping yang benar-benar
diikuti penurunan harga"). Audit sebelumnya sudah membuktikan topping mentah (tanpa gate)
EV-nya negatif; pertanyaan di sini: **apakah menambah confirmation + regime + ML mengubah itu**,
persis seperti gate BUY berubah dari tidak-signifikan (reversal saja) menjadi signifikan
(reversal + regime, Z=1.97)?

In [8]:
def add_topping_confirmation(feature_frame: pd.DataFrame,
                              high: pd.Series, low: pd.Series, close: pd.Series) -> pd.DataFrame:
    """Analog add_reversal_confirmation, tapi arah BEARISH (konfirmasi topping, bukan bottoming).

    - confirm_lower_close: close hari ini < close kemarin (harga mulai turun)
    - confirm_break_low:   close hari ini < low kemarin (menembus support harian)
    - confirm_rsi_falling: RSI hari ini < RSI kemarin (momentum berbalik turun)
    - confirm_bearish_div: harga bikin higher high tapi RSI bikin lower high (divergence)
    - confirm_not_pumping: bukan higher high 2 hari beruntun (euphoria berhenti)

    topping_confirmed: True hanya jika topping AND minimal 2 dari 5 konfirmasi terpenuhi.
    """
    df = feature_frame.copy()
    rsi_series = df.get("rsi", pd.Series(float("nan"), index=df.index))

    close = close.reindex(df.index)
    high = high.reindex(df.index)
    low = low.reindex(df.index)

    prev_close = close.shift(1)
    prev_high = high.shift(1)
    prev_low = low.shift(1)
    prev_rsi = rsi_series.shift(1)

    confirm_lower_close = close < prev_close
    confirm_break_low = close < prev_low
    confirm_rsi_falling = rsi_series < prev_rsi
    confirm_bearish_div = (high > prev_high) & (rsi_series < prev_rsi)
    confirm_not_pumping = ~((high > prev_high) & (prev_high > high.shift(2)))

    df["confirm_lower_close"] = confirm_lower_close.fillna(False)
    df["confirm_break_low"] = confirm_break_low.fillna(False)
    df["confirm_rsi_falling"] = confirm_rsi_falling.fillna(False)
    df["confirm_bearish_div"] = confirm_bearish_div.fillna(False)
    df["confirm_not_pumping"] = confirm_not_pumping.fillna(False)

    confirm_count_sell = (
        df["confirm_lower_close"].astype(int)
        + df["confirm_break_low"].astype(int)
        + df["confirm_rsi_falling"].astype(int)
        + df["confirm_bearish_div"].astype(int)
        + df["confirm_not_pumping"].astype(int)
    )
    df["confirm_count_sell"] = confirm_count_sell

    topping = df.get("topping", pd.Series(False, index=df.index))
    df["topping_confirmed"] = topping & (confirm_count_sell >= 2)

    return df


def add_regime_filter_sell(feature_frame: pd.DataFrame, close: pd.Series,
                            ma_window: int = 200, pump_lookback: int = 5,
                            pump_threshold: float = 0.15, stable_days: int = 2) -> pd.DataFrame:
    """Analog add_regime_filter, tapi mendeteksi euphoria/pump berlebihan (kebalikan crash).

    - pumping:        naik > pump_threshold dalam pump_lookback hari terakhir (euphoria tajam)
    - price_topped:   harga tidak bikin higher high >= stable_days hari (mulai melemah)
    - regime_ok_sell: AMAN untuk topping entry: SEDANG pumping DAN harga sudah mulai topped
                       (analog 'regime_ok' di BUY: bukan crashing DAN sudah stabil -- di sini
                       dibalik jadi: SEDANG pumping kuat DAN momentum sudah mulai melemah)
    """
    df = feature_frame.copy()
    close = close.reindex(df.index)

    roll_return = close.pct_change(pump_lookback)
    df["pumping"] = (roll_return > pump_threshold).fillna(False)

    is_higher_high = close > close.shift(1)
    higher_high_streak = is_higher_high.rolling(stable_days, min_periods=stable_days).sum()
    df["price_topped"] = (higher_high_streak < stable_days).fillna(False)

    df["regime_ok_sell"] = df["pumping"] & df["price_topped"]

    return df


print("Sell-side gate functions ready.")

Sell-side gate functions ready.


In [9]:
feats_sell = {}
for s, d in all_dfs.items():
    feat = build_feature_frame(d["close"], d["volume"], btc_df["close"], fng)
    feat = compute_reversal_scores(feat)
    feat = add_topping_confirmation(feat, d["high"], d["low"], d["close"])
    feat = add_regime_filter_sell(feat, d["close"])
    feat = add_forward_return_target(feat, d["close"], horizon=v5.FORWARD_HORIZON)
    feats_sell[s] = feat

pool_sell = pd.concat([f.assign(symbol=s) for s, f in feats_sell.items()])
pool_sell = pool_sell[pool_sell["forward_return"].notna()]
print(f"Total observasi (semua coin): {len(pool_sell):,}")


def eval_sell_condition(mask, name):
    """Z-test + EV untuk kondisi SELL: menang kalau forward_return < 0 (harga turun)."""
    sel = pool_sell[mask & pool_sell["forward_return"].notna()]
    n = len(sel)
    if n == 0:
        print(f"{name}: n=0")
        return None
    wins = (sel["forward_return"] < 0).sum()
    rets = -sel["forward_return"]  # balik tanda supaya "profit" positif kalau harga turun
    p_obs, z, p_val = ztest(wins, n)
    win_r = rets[rets > 0]
    loss_r = rets[rets <= 0]
    avg_w = win_r.mean() if len(win_r) else 0.0
    avg_l = abs(loss_r.mean()) if len(loss_r) else 0.0
    ev = (p_obs * avg_w) - ((1 - p_obs) * avg_l)
    significant = abs(z) > 1.96
    enough_sample = n >= 100
    verdict = ("SAMPLE < 100" if not enough_sample else
               "VALID + EV+" if (significant and ev > 0) else
               "signif tapi EV-" if significant else "TIDAK SIGNIFIKAN")
    print(f"{name:38s} n={n:5d}  win_rate={p_obs:6.1%}  Z={z:+6.2f}  p={p_val:7.4f}  "
          f"EV={ev:+7.3%}  {verdict}")
    return {"name": name, "n": n, "win_rate": float(p_obs), "z": float(z),
            "p_value": float(p_val), "ev": float(ev), "significant": bool(significant),
            "enough_sample": bool(enough_sample), "verdict": verdict}


print(f"\n{'Kondisi':38s} {'n':>6s}  {'WinRate':>7s}  {'Z':>7s}  {'p-val':>8s}  {'EV':>8s}  Verdict")
print("-" * 100)
r1 = eval_sell_condition(pool_sell["topping"], "TOPPING (RSI>70 & FGI>75)")
r2 = eval_sell_condition(pool_sell["topping"] & pool_sell["topping_confirmed"],
                          "TOPPING + confirmed")
r3 = eval_sell_condition(
    pool_sell["topping"] & pool_sell["topping_confirmed"] & pool_sell["regime_ok_sell"],
    "TOPPING + confirmed + regime_sell")

Total observasi (semua coin): 20,264

Kondisi                                     n  WinRate        Z     p-val        EV  Verdict
----------------------------------------------------------------------------------------------------
TOPPING (RSI>70 & FGI>75)              n=  720  win_rate= 39.7%  Z= -5.52  p= 0.0000  EV=-10.385%  signif tapi EV-
TOPPING + confirmed                    n=  178  win_rate= 38.8%  Z= -3.00  p= 0.0027  EV=-8.212%  signif tapi EV-
TOPPING + confirmed + regime_sell      n=   81  win_rate= 32.1%  Z= -3.22  p= 0.0013  EV=-11.419%  SAMPLE < 100


Hasil gate bertahap SELL **kebalikan pola dari BUY**: di BUY, menambah gate (reversal_confirmed
→ regime_ok) memperbaiki hasil (Z naik dari 0.54 jadi 1.97, EV naik dari +0.46% jadi +1.70%).
Di SELL, menambah gate yang sama justru **memburuk di tahap akhir** (EV -10.4% → -8.2% →
-11.4%), dan n jatuh di bawah 100 pada gate penuh — sample terlalu kecil untuk dipercaya
meski hasilnya sudah negatif.

Tetap lanjutkan ke lapisan ML (RandomForest, arsitektur sama dengan `v5.train_model`) untuk
memberi kesempatan penuh — barangkali kombinasi fitur non-linear menemukan pola yang tidak
tertangkap gate biner di atas (ini juga alasan kenapa ablasi fitur BUY sebelumnya bisa
berbeda dari korelasi Pearson sederhana).

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

# Fitur SELL: analog FEATURE_COLS BUY, tapi confirm_count_sell menggantikan confirm_count
SELL_FEATURE_COLS = ["rsi", "volume_ratio", "fng_value", "relative_strength",
                     "confirm_count_sell", "ma_diff_pct"]

candidates_sell = pool_sell[pool_sell["topping"] & pool_sell["forward_return"].notna()].copy()
candidates_sell = candidates_sell.dropna(subset=SELL_FEATURE_COLS)
print(f"Total sinyal topping (kandidat training): {len(candidates_sell)}")

train_sell = candidates_sell[candidates_sell.index < pd.Timestamp("2024-01-01")]
test_sell = candidates_sell[candidates_sell.index >= pd.Timestamp("2024-01-01")]
print(f"In-sample (<2024): {len(train_sell)}   Out-sample (>=2024): {len(test_sell)}")

if len(train_sell) >= 50 and len(test_sell) >= 20:
    X_train = train_sell[SELL_FEATURE_COLS].values
    y_train = (train_sell["forward_return"] < 0).astype(int).values  # 1 = harga turun (menang utk SELL)
    X_test = test_sell[SELL_FEATURE_COLS].values
    y_test = (test_sell["forward_return"] < 0).astype(int).values

    scaler_sell = StandardScaler().fit(X_train)
    model_sell = RandomForestClassifier(**v5.RF_PARAMS)
    model_sell.fit(scaler_sell.transform(X_train), y_train)

    in_acc = model_sell.score(scaler_sell.transform(X_train), y_train)
    out_acc = model_sell.score(scaler_sell.transform(X_test), y_test)
    print(f"\nWalk-forward (arsitektur sama dgn strategy_v5.train_model):")
    print(f"  In-sample (<2024):  {in_acc:.1%} ({len(train_sell)} sinyal)")
    print(f"  Out-sample (>=2024): {out_acc:.1%} ({len(test_sell)} sinyal)")
    print(f"  Gap: {in_acc - out_acc:+.1%}")
    print(f"  (baseline acak/naive: 50%, atau base rate turun = {(candidates_sell['forward_return']<0).mean():.1%})")

    # 5-fold CV di gabungan semua data topping
    X_all = candidates_sell[SELL_FEATURE_COLS].values
    y_all = (candidates_sell["forward_return"] < 0).astype(int).values
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []
    for tr_idx, te_idx in kf.split(X_all):
        sc = StandardScaler().fit(X_all[tr_idx])
        m = RandomForestClassifier(**v5.RF_PARAMS)
        m.fit(sc.transform(X_all[tr_idx]), y_all[tr_idx])
        fold_scores.append(m.score(sc.transform(X_all[te_idx]), y_all[te_idx]))
    print(f"  5-fold CV: {np.mean(fold_scores):.1%} (+/-{np.std(fold_scores):.1%})")
else:
    print("Sample terlalu kecil untuk walk-forward yang berarti (butuh >=50 in-sample, >=20 out-sample).")

Total sinyal topping (kandidat training): 581
In-sample (<2024): 208   Out-sample (>=2024): 373

Walk-forward (arsitektur sama dgn strategy_v5.train_model):
  In-sample (<2024):  59.6% (208 sinyal)
  Out-sample (>=2024): 60.3% (373 sinyal)
  Gap: -0.7%
  (baseline acak/naive: 50%, atau base rate turun = 39.9%)


  5-fold CV: 60.9% (+/-6.7%)


**Waspada base rate** (Pilar 1 — jangan tertipu akurasi mentah): base rate "harga turun" di
kandidat topping cuma **39.9%** — model yang SELALU memprediksi "tidak turun" (kelas mayoritas)
otomatis dapat akurasi ~60% tanpa belajar apa-apa. Akurasi out-sample 60.3% di atas TIDAK
otomatis berarti model punya edge; perlu cek precision & EV spesifik pada prediksi
"akan turun" (kelas minoritas, yang justru relevan untuk sinyal SELL), bukan overall accuracy.

In [11]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

y_pred_test = model_sell.predict(scaler_sell.transform(X_test))
cm = confusion_matrix(y_test, y_pred_test)
print("Confusion matrix (out-sample, baris=aktual, kolom=prediksi) [0=naik/flat, 1=turun]:")
print(cm)

precision_turun = precision_score(y_test, y_pred_test, pos_label=1, zero_division=0)
recall_turun = recall_score(y_test, y_pred_test, pos_label=1, zero_division=0)
n_predicted_turun = int((y_pred_test == 1).sum())
print(f"\nModel memprediksi 'akan turun' pada {n_predicted_turun} dari {len(y_test)} kasus out-sample.")
print(f"Precision kelas 'turun' (dari yang diprediksi turun, berapa % benar turun): {precision_turun:.1%}")
print(f"Recall kelas 'turun' (dari yang benar turun, berapa % tertangkap): {recall_turun:.1%}")
print(f"Base rate turun di out-sample: {y_test.mean():.1%}")

if n_predicted_turun > 0:
    # EV pada subset yang model prediksi "akan turun" -- ini yang relevan buat sinyal SELL beneran
    mask_pred_turun = y_pred_test == 1
    subset = test_sell.iloc[mask_pred_turun]
    rets = -subset["forward_return"]  # profit positif kalau harga beneran turun
    win_rate_ml = (rets > 0).mean()
    ev_ml = rets.mean()
    p_obs, z, p_val = ztest(int((rets > 0).sum()), len(rets))
    print(f"\nEV pada {len(subset)} kasus yang model beri sinyal SELL (ml_confidence tinggi):")
    print(f"  win_rate={win_rate_ml:.1%}  Z={z:+.2f}  p={p_val:.4f}  EV={ev_ml:+.3%}")
else:
    print("\nModel TIDAK PERNAH memprediksi 'akan turun' di out-sample -- artinya model")
    print("cuma belajar 'selalu prediksi tidak turun' (memanfaatkan base rate), bukan edge asli.")

Confusion matrix (out-sample, baris=aktual, kolom=prediksi) [0=naik/flat, 1=turun]:
[[225   0]
 [148   0]]

Model memprediksi 'akan turun' pada 0 dari 373 kasus out-sample.
Precision kelas 'turun' (dari yang diprediksi turun, berapa % benar turun): 0.0%
Recall kelas 'turun' (dari yang benar turun, berapa % tertangkap): 0.0%
Base rate turun di out-sample: 39.7%

Model TIDAK PERNAH memprediksi 'akan turun' di out-sample -- artinya model
cuma belajar 'selalu prediksi tidak turun' (memanfaatkan base rate), bukan edge asli.


## 8. Kesimpulan bagian 7 (gate SELL simetris + ML)

**Confusion matrix membuktikan model TIDAK belajar apa-apa**: dari 373 kasus out-sample,
model **0 kali** memprediksi "akan turun" — dia selalu memilih kelas mayoritas ("tidak turun",
base rate 60.1%). Akurasi 60.3% yang di bagian 7 tadi terlihat bagus (gap walk-forward malah
negatif -0.7%, kelihatan "generalisasi baik") ternyata murni **ilusi base rate** — model
RandomForest ini gagal menemukan pola apapun yang membedakan topping yang diikuti penurunan
vs yang tidak, dan solusi "termudah" secara matematis (minimalkan loss) adalah menyerah dan
selalu prediksi kelas mayoritas.

**Ini konsisten dengan seluruh rangkaian temuan sesi ini**:

| Pendekatan | Hasil |
|---|---|
| Topping mentah sebagai entry SHORT (audit awal) | EV -10.4% |
| Topping sebagai exit trigger dari posisi BUY | n=0, tidak mungkin terjadi dalam horizon |
| Exit dini berbasis RSI-weakening (12 varian threshold) | Semua signifikan LEBIH BURUK dari hold |
| Gate SELL simetris: topping + confirmed + regime_sell | EV memburuk di tiap tahap (-10.4% → -8.2% → -11.4%), n<100 di gate penuh |
| Gate SELL + ML (RandomForest, arsitektur sama dgn BUY) | Model gagal belajar, selalu prediksi mayoritas |

**Kesimpulan akhir yang jujur**: sisi SELL/short di crypto (untuk 10 coin & periode data ini)
**tidak menunjukkan edge statistik yang bisa dipercaya**, dengan pendekatan apapun yang dicoba
sejauh ini — baik sebagai exit signal, entry short berbasis rule, maupun ML. Ini BUKAN karena
kurang usaha; justru sebaliknya, ini bukti bahwa crypto punya **asimetri struktural**: sisi
BUY (bottoming/mean-reversion dari oversold) punya edge yang terbukti robust, sedangkan sisi
SELL (topping/momentum reversal dari overbought) tidak — karena momentum naik di crypto
cenderung berkepanjangan (persis dibahas di dokumen `Time Horizon Crypto` & `Probability
Theory & Random Walk`: trend yang terbentuk dari euphoria retail lebih persisten daripada
trend yang terbentuk dari capitulation/panic selling).

**Rekomendasi**: JANGAN paksakan mengaktifkan sinyal SELL/short di V5 dengan data & metodologi
saat ini. Opsi yang tersisa dan belum diuji (di luar scope sesi ini): (a) trailing-stop
berbasis ATR untuk mengunci profit tanpa perlu prediksi arah, (b) memperbesar sample dengan
lebih banyak coin/periode data sebelum mencoba lagi gate SELL berbasis ML.

## 9. Catatan penting: ini bukan murni "kurang data", tapi "kurang kejadian target"

Dataset yang dipakai (2018–2026, 10 coin, 20.264 observasi) sebenarnya besar dari segi jumlah
baris. Yang membuat sisi SELL gagal bukan jumlah data secara umum, tapi **base rate kejadian
targetnya sendiri jarang**: dari semua kondisi topping, cuma 39.9% yang benar-benar diikuti
harga turun (bagian 7). Ini bukan artefak sampling/bug, tapi refleksi periode data yang
didominasi bull market crypto (BTC $300 → puluhan ribu dolar sepanjang 2018–2026) — momen
crash besar (bear 2018, Luna/Terra Mei 2022, FTX Nov 2022) proporsinya kecil dibanding total
hari observasi, jadi jumlah kejadian "topping yang benar-benar diikuti crash" jadi sedikit
meski total baris datanya banyak.

**Status: sinyal SELL/short DITUNDA**, bukan ditolak permanen. Layak dicoba ulang nanti kalau:
- Dataset diperluas mencakup lebih banyak periode bear market secara proporsional, atau
- Ada coin/pair tambahan yang historisnya mencakup crash lebih banyak, atau
- Horizon & definisi target diubah untuk menangkap capitulation/crash jangka menengah
  (bukan 5 hari) — sesuai temuan bagian 2 bahwa reversal besar butuh waktu jauh lebih lama.

Tidak ada perubahan ke `strategy_v5.py` produksi dari riset ini.